Balanced Dragonfly
number of switches, servers (endpoints), links, and per-switch port count (radix)

In [28]:
def closest_dragonfly_weighted_links_r(S_target=None, T_target=None, Lss_target=None,
                                       r_max=64, a_max=64, weights=(1.0, 1.0, 1.0)):
    """
    Find Dragonfly config closest to target with constraint a = 2*p = 2*h
    using weighted error: weights = (wS, wT, wL)
    """
    best = None
    best_err = float('inf')
    wS, wT, wL = weights

    for a in range(2, a_max+1):
        p = h = a // 2  # enforce a = 2*p = 2*h
        r = p + h + (a - 1)
        if r > r_max:
            continue  # exceeds port count per switch

        # number of switches
        g = S_target // a if S_target else a
        S = a * g

        # total servers
        T = S * p

        # switch-to-switch links (approximation)
        Lss = g * a * (a - 1) // 2 + g * h

        # weighted error
        err = 0.0
        if S_target:
            err += wS * ((S - S_target)/S_target)**2
        if T_target:
            err += wT * ((T - T_target)/T_target)**2
        if Lss_target:
            err += wL * ((Lss - Lss_target)/Lss_target)**2

        if err < best_err:
            best_err = err
            best = dict(a=a, h=h, p=p, g=g, S=S, T=T, Lss=Lss, r=r, err=err)

    return best

best = closest_dragonfly_weighted_links_r(S_target=80, T_target=3072, Lss_target=1024,
                                          r_max=64, a_max=64, weights=(1.0, 1.0, 0.5))
print(best)

{'a': 26, 'h': 13, 'p': 13, 'g': 3, 'S': 78, 'T': 1014, 'Lss': 1014, 'r': 51, 'err': 0.4494680023193359}


Closest Dragonfly

Parameters

S_target

Target number of switches you want to approximate with a Dragonfly.

Example: if your topology has 512 switches, you’d set S_target=512.

T_target

Target number of servers (endpoints) you want the Dragonfly to host.

Example: if your network connects 8192 servers, set T_target=8192.

Lss_target

Target number of switch–switch links (intra + inter group).

Lets you capture link budget so that comparisons don’t mismatch connectivity.

r_max (default 64)

Maximum switch radix (number of ports per switch).

Reflects hardware constraint, e.g. if you’re comparing against a 64-port switch ASIC.

a_max (default 64)

Maximum number of switches per group (a) to consider when searching.

Caps the search space since Dragonfly can scale very large otherwise.

weights (default (1.0, 1.0, 0.5))

Weights for the cost function when comparing to your target.

Order is (switches, servers, links).

Example: (1.0, 1.0, 0.5) means matching number of switches and servers is twice as important as matching number of links.

You can tune these depending on what you care about more.

In [32]:
def closest_dragonfly(S_target=None, T_target=None, Lss_target=None, 
                      r_max=64, a_max=64, weights=(1.0, 1.0, 0.5), top_k=10):
    """
    Find top-k Dragonfly configurations closest to target specs.
    """

    candidates = []

    for a in range(2, S_max+1):       # routers per group
        for h in range(1, a):         # inter-group links per router
            for r in range(1,r_max+1):
                # r = r_max
                p = r - (a - 1) - h       # ports for servers
                if p <= 0:
                    continue
                g = a * h + 1             # groups
                S = a * g                 # total switches
                T = p * S                 # total servers
                Lss = (a * (a - 1) // 2) * g  # total intra-group links

                # Compute error metric
                err = 0.0
                if S_target:
                    err += weights[0] * abs(S - S_target) / S_target
                if T_target:
                    err += weights[1] * abs(T - T_target) / T_target
                if Lss_target:
                    err += weights[2] * abs(Lss - Lss_target) / Lss_target

                candidates.append({
                    'a': a, 'h': h, 'p': p, 'g': g, 
                    'S': S, 'T': T, 'Lss': Lss, 'r': r, 'err': err
                })

    # Sort candidates by error and return top-k
    candidates.sort(key=lambda x: x['err'])
    return candidates[:top_k]

best = closest_dragonfly(S_target=80, T_target=3072, Lss_target=1024, r_max=64)
print(best)

[{'a': 5, 'h': 3, 'p': 38, 'g': 16, 'S': 80, 'T': 3040, 'Lss': 160, 'r': 45, 'err': 0.4322916666666667}, {'a': 5, 'h': 3, 'p': 39, 'g': 16, 'S': 80, 'T': 3120, 'Lss': 160, 'r': 46, 'err': 0.4375}, {'a': 6, 'h': 2, 'p': 39, 'g': 13, 'S': 78, 'T': 3042, 'Lss': 195, 'r': 46, 'err': 0.43955078125}, {'a': 6, 'h': 2, 'p': 40, 'g': 13, 'S': 78, 'T': 3120, 'Lss': 195, 'r': 47, 'err': 0.44541015625}, {'a': 9, 'h': 1, 'p': 34, 'g': 10, 'S': 90, 'T': 3060, 'Lss': 360, 'r': 43, 'err': 0.453125}, {'a': 5, 'h': 3, 'p': 37, 'g': 16, 'S': 80, 'T': 2960, 'Lss': 160, 'r': 44, 'err': 0.4583333333333333}, {'a': 5, 'h': 3, 'p': 40, 'g': 16, 'S': 80, 'T': 3200, 'Lss': 160, 'r': 47, 'err': 0.4635416666666667}, {'a': 6, 'h': 2, 'p': 38, 'g': 13, 'S': 78, 'T': 2964, 'Lss': 195, 'r': 45, 'err': 0.46494140625}, {'a': 6, 'h': 2, 'p': 41, 'g': 13, 'S': 78, 'T': 3198, 'Lss': 195, 'r': 48, 'err': 0.47080078125}, {'a': 9, 'h': 1, 'p': 35, 'g': 10, 'S': 90, 'T': 3150, 'Lss': 360, 'r': 44, 'err': 0.474609375}]


weights=(1.0, 1.0, 0.5)

{'a': 7, 'h': 1, 'p': 57, 'g': 8, 'S': 56, 'T': 3192, 'Lss': 196, 'r': 64, 'err': 0.7912109375}

{
  'a': 7,    # switches per group
  'h': 1,    # servers per switch
  'p': 57,   # intra-group ports per switch
  'g': 8,    # number of groups
  'S': 56,   # total switches
  'T': 3192, # total servers
  'Lss': 196,# total switch-to-switch links
  'r': 64,   # switch radix (ports per switch)
  'err': 0.7912109375 # cost function error
}

[{'a': 7, 'h': 1, 'p': 57, 'g': 8, 'S': 56, 'T': 3192, 'Lss': 168, 'r': 64, 'err': 0.798046875}, 

{'a': 5, 'h': 2, 'p': 58, 'g': 11, 'S': 55, 'T': 3190, 'Lss': 110, 'r': 64, 'err': 0.8240559895833333}, 

{'a': 4, 'h': 3, 'p': 58, 'g': 13, 'S': 52, 'T': 3016, 'Lss': 78, 'r': 64, 'err': 0.8491861979166666}, 

{'a': 8, 'h': 1, 'p': 56, 'g': 9, 'S': 72, 'T': 4032, 'Lss': 252, 'r': 64, 'err': 0.8509765625}, 

{'a': 6, 'h': 2, 'p': 57, 'g': 13, 'S': 78, 'T': 4446, 'Lss': 195, 'r': 64, 'err': 0.924658203125}, 

{'a': 5, 'h': 3, 'p': 57, 'g': 16, 'S': 80, 'T': 4560, 'Lss': 160, 'r': 64, 'err': 0.9453125}, 

{'a': 9, 'h': 1, 'p': 55, 'g': 10, 'S': 90, 'T': 4950, 'Lss': 360, 'r': 64, 'err': 1.1484375}, 

{'a': 6, 'h': 1, 'p': 58, 'g': 7, 'S': 42, 'T': 2436, 'Lss': 105, 'r': 64, 'err': 1.156396484375}, 

{'a': 4, 'h': 2, 'p': 59, 'g': 9, 'S': 36, 'T': 2124, 'Lss': 54, 'r': 64, 'err': 1.34541015625}, 

{'a': 5, 'h': 1, 'p': 59, 'g': 6, 'S': 30, 'T': 1770, 'Lss': 60, 'r': 64, 'err': 1.5341796875}]

In [30]:
def closest_dragonfly(S_target=None, T_target=None, Lss_target=None, 
                      r_max=64, a_max=64, weights=(1.0, 0, 1.0), top_k=10):
    """
    Find top-k Dragonfly configurations closest to target specs.
    """

    candidates = []

    for a in range(2, S_max+1):       # routers per group
        for h in range(1, a):         # inter-group links per router
            r = r_max
            p = r - (a - 1) - h       # ports for servers
            if p <= 0:
                continue
            g = a * h + 1             # groups
            S = a * g                 # total switches
            T = p * S                 # total servers
            Lss = (a * (a - 1) // 2) * g  # total intra-group links

            # Compute error metric
            err = 0.0
            if S_target:
                err += weights[0] * abs(S - S_target) / S_target
            if T_target:
                err += weights[1] * abs(T - T_target) / T_target
            if Lss_target:
                err += weights[2] * abs(Lss - Lss_target) / Lss_target

            candidates.append({
                'a': a, 'h': h, 'p': p, 'g': g, 
                'S': S, 'T': T, 'Lss': Lss, 'r': r, 'err': err
            })

    # Sort candidates by error and return top-k
    candidates.sort(key=lambda x: x['err'])
    return candidates[:top_k]

best = closest_dragonfly(S_target=80, T_target=3072, Lss_target=1024, r_max=64)
print(best)

[{'a': 9, 'h': 1, 'p': 55, 'g': 10, 'S': 90, 'T': 4950, 'Lss': 360, 'r': 64, 'err': 0.7734375}, {'a': 6, 'h': 2, 'p': 57, 'g': 13, 'S': 78, 'T': 4446, 'Lss': 195, 'r': 64, 'err': 0.8345703125}, {'a': 5, 'h': 3, 'p': 57, 'g': 16, 'S': 80, 'T': 4560, 'Lss': 160, 'r': 64, 'err': 0.84375}, {'a': 8, 'h': 1, 'p': 56, 'g': 9, 'S': 72, 'T': 4032, 'Lss': 252, 'r': 64, 'err': 0.85390625}, {'a': 10, 'h': 1, 'p': 54, 'g': 11, 'S': 110, 'T': 5940, 'Lss': 495, 'r': 64, 'err': 0.8916015625}, {'a': 7, 'h': 2, 'p': 56, 'g': 15, 'S': 105, 'T': 5880, 'Lss': 315, 'r': 64, 'err': 1.0048828125}, {'a': 11, 'h': 1, 'p': 53, 'g': 12, 'S': 132, 'T': 6996, 'Lss': 660, 'r': 64, 'err': 1.00546875}, {'a': 5, 'h': 4, 'p': 56, 'g': 21, 'S': 105, 'T': 5880, 'Lss': 210, 'r': 64, 'err': 1.107421875}, {'a': 12, 'h': 1, 'p': 52, 'g': 13, 'S': 156, 'T': 8112, 'Lss': 858, 'r': 64, 'err': 1.112109375}, {'a': 7, 'h': 1, 'p': 57, 'g': 8, 'S': 56, 'T': 3192, 'Lss': 168, 'r': 64, 'err': 1.1359375}]


[{'a': 5, 'h': 3, 'p': 57, 'g': 16, 'S': 80, 'T': 4560, 'Lss': 160, 'r': 64, 'err': 0.921875}, 

{'a': 6, 'h': 2, 'p': 57, 'g': 13, 'S': 78, 'T': 4446, 'Lss': 195, 'r': 64, 'err': 0.92978515625}, 

{'a': 9, 'h': 1, 'p': 55, 'g': 10, 'S': 90, 'T': 4950, 'Lss': 360, 'r': 64, 'err': 0.94921875}, 

{'a': 8, 'h': 1, 'p': 56, 'g': 9, 'S': 72, 'T': 4032, 'Lss': 252, 'r': 64, 'err': 0.976953125}, 

{'a': 10, 'h': 1, 'p': 54, 'g': 11, 'S': 110, 'T': 5940, 'Lss': 495, 'r': 64, 'err': 1.13330078125}, 

{'a': 7, 'h': 2, 'p': 56, 'g': 15, 'S': 105, 'T': 5880, 'Lss': 315, 'r': 64, 'err': 1.15869140625}, 

{'a': 5, 'h': 4, 'p': 56, 'g': 21, 'S': 105, 'T': 5880, 'Lss': 210, 'r': 64, 'err': 1.2099609375}, 

{'a': 7, 'h': 1, 'p': 57, 'g': 8, 'S': 56, 'T': 3192, 'Lss': 168, 'r': 64, 'err': 1.21796875}, 

{'a': 5, 'h': 2, 'p': 58, 'g': 11, 'S': 55, 'T': 3190, 'Lss': 110, 'r': 64, 'err': 1.2587890625}, 

{'a': 6, 'h': 3, 'p': 56, 'g': 19, 'S': 114, 'T': 6384, 'Lss': 285, 'r': 64, 'err': 1.28583984375}]

In [34]:
def max_servers_dragonfly(max_switches, max_switch_links, r_max_input=64, a_max=64, h_max=None):
    """
    Find the Dragonfly configuration with maximum servers
    without exceeding max_switches and max switch-to-switch links.
    
    Parameters:
    - max_switches: maximum number of switches
    - max_switch_links: maximum number of switch-to-switch links
    - r_max: maximum ports per switch
    - a_max: maximum number of switches per group
    - h_max: maximum inter-group links per switch (default: r_max - (a-1))
    
    Returns:
    - dict with best Dragonfly config and max servers
    """
    best = None
    max_servers_supported = 0

    for a in range(2, min(a_max, max_switches)+1):
        for r_max in range(1, r_max_input+1):
            # number of switches per group
            max_h = r_max - (a-1)
            h_limit = min(h_max if h_max is not None else max_h, max_h)
            for h in range(1, h_limit+1):
                # total switches
                g = max_switches // a
                S = a * g
                if S > max_switches:
                    continue
                # switch-to-switch links
                intra_links = a*(a-1)//2 * g
                inter_links = g*(g-1)//2 * a*h
                total_switch_links = intra_links + inter_links
                if total_switch_links > max_switch_links:
                    continue
                # ports left for servers
                p = r_max - (a-1) - h
                if p <= 0:
                    continue
                # total servers
                T = S * p
                if T > max_servers_supported:
                    max_servers_supported = T
                    best = dict(a=a, h=h, g=g, S=S, p=p, T=T, r=r_max,
                                intra_links=intra_links,
                                inter_links=inter_links,
                                total_switch_links=total_switch_links)
    return best

max_switches = 80
max_links = 1024
r_max = 64

best_config = max_servers_dragonfly(max_switches, max_links, r_max_input=r_max)
print(best_config)

{'a': 4, 'h': 1, 'g': 20, 'S': 80, 'p': 60, 'T': 4800, 'r': 64, 'intra_links': 120, 'inter_links': 760, 'total_switch_links': 880}


My own algorithm:
- Constraint 1: cannot exceed the port count
- Constraint 2: cannot have less than 100% of the servers
- Constraint 3: favor more inter-group links
- Constraint 4: the ratio a/h is not ridiculous
- Constraint 5: a >= h (suggestion is a = 2*h)
- Constraint 6: cannot exceed the number of switches

In [10]:
def find_dragonfly(max_switches, max_links, min_servers, max_ports):
    # a: num switches per group
    # h: inter-group links per switch
    # p: num servers per switch

    best = None
    largest_ratio = 0

    min_a = 2
    for a in range(min_a, max_switches+1): # constraint 4
        for r in range(2, max_ports+1): # constraint 1
            for h in range(1,r-(a-1)): # need to connect to at least 1 server
                if a < h: # constraint 5
                    continue

                num_switches = a * (a*h + 1)
                if num_switches > max_switches: # constraint 6
                    continue

                g = num_switches // a

                intra_links = a*(a-1)//2 * g
                inter_links = g*(g-1)//2 # g*(g-1)//2 * a*h
                num_links = intra_links + inter_links
                if num_links > max_links:
                    continue
                
                for p in range(1, r - (a-1) - h+1):
                    num_servers = a * p * g
                    if num_servers < min_servers: # constraint 2
                        continue

                    # print(f"a={a}, h={h}, r={r}, g={g}, S={num_switches}, p={p}, T={num_servers}, links={num_links}\n")
                    ratio = h/p
                    # ratio = h
                    if ratio > largest_ratio: # constrant 3
                        largest_ratio = ratio
                        best = dict(a=a, r=r, h=h, g=g, p=p, num_switches=num_switches, num_links=num_links, num_servers=num_servers)

    return best

best = find_dragonfly(80, 1024, 3072, 64)
print(best)

{'a': 4, 'r': 53, 'h': 4, 'g': 17, 'p': 46, 'num_switches': 68, 'num_links': 238, 'num_servers': 3128}


Fix the r and p range:

no constraint 5: {'a': 2, 'r': 60, 'h': 19, 'g': 39, 'p': 40, 'num_switches': 78, 'num_links': 780, 'num_servers': 3120}
constraint 5: {'a': 4, 'r': 53, 'h': 4, 'g': 17, 'p': 46, 'num_switches': 68, 'num_links': 238, 'num_servers': 3128}


----------------------
Add constraint 5 or not:

ratio = h/p or h:

Add min_a or not:



(no constraint 5, ratio h, min_a=2: {'a': 2, 'r': 61, 'h': 19, 'g': 39, 'p': 40, 'num_switches': 78, 'num_links': 780, 'num_servers': 3120})
(no contraint 5, ratio h, min_a=5: {'a': 5, 'r': 47, 'h': 3, 'g': 16, 'p': 39, 'num_switches': 80, 'num_links': 280, 'num_servers': 3120})
no constraint 5, ratio h/p, min_a = 2: {'a': 2, 'r': 61, 'h': 19, 'g': 39, 'p': 40, 'num_switches': 78, 'num_links': 780, 'num_servers': 3120}
no constraint 5, ratio h/p, min_a = 5: {'a': 5, 'r': 47, 'h': 3, 'g': 16, 'p': 39, 'num_switches': 80, 'num_links': 280, 'num_servers': 3120}
constraint 5, ratio h/p, min_a = 2: {'a': 4, 'r': 54, 'h': 4, 'g': 17, 'p': 46, 'num_switches': 68, 'num_links': 238, 'num_servers': 3128}

DISCARD

min_a = 2:

ratio = h/p: {'a': 2, 'r': 64, 'h': 23, 'g': 40, 'p': 39, 'num_switches': 80, 'num_links': 820, 'num_servers': 3120}

ratio = h: {'a': 2, 'r': 64, 'h': 23, 'g': 40, 'p': 39, 'num_switches': 80, 'num_links': 820, 'num_servers': 3120}

min_a = 3:

ratio = h/p: {'a': 4, 'r': 64, 'h': 21, 'g': 20, 'p': 39, 'num_switches': 80, 'num_links': 310, 'num_servers': 3120}

ratio = h: {'a': 3, 'r': 64, 'h': 21, 'g': 26, 'p': 40, 'num_switches': 78, 'num_links': 403, 'num_servers': 3120}

min_a = 5:

ratio = h/p: {'a': 5, 'r': 64, 'h': 20, 'g': 16, 'p': 39, 'num_switches': 80, 'num_links': 280, 'num_servers': 3120}

----------------

adding in constraint 5: {'a': 11, 'r': 62, 'h': 11, 'g': 7, 'p': 40, 'num_switches': 77, 'num_links': 406, 'num_servers': 3080}